In [0]:
jdbc_url = (
    "jdbc:sqlserver://capstone3server.database.windows.net:1433;"
    "database=capstone3;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

try:
    df = (spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "dbo.airline_id")
        .option("user", "ltmadmin")
        .option("password", "P@55word")
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load())
    display(df)
except Exception as e:
    print(str(e)[:2000])

In [0]:
jdbc_url = (
    "jdbc:sqlserver://capstone3server.database.windows.net:1433;"
    "database=capstone3;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

try:
    df1 = (spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "dbo.airport_id")
        .option("user", "ltmadmin")
        .option("password", "P@55word")
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load())
    display(df1)
except Exception as e:
    print(str(e)[:2000])

In [0]:
# Save airline data
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("capstone_3flight.flight_schema.airline_id")


# Save airport data
df1.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("capstone_3flight.flight_schema.airport_id")

print("Both tables successfully saved as Delta tables!")

In [0]:
# Read saved Delta tables
saved_airline_df = spark.table(
    "capstone_3flight.flight_schema.airline"
)

saved_airport_df = spark.table(
    "capstone_3flight.flight_schema.airport"
)

# Calculate final row counts
airline_count = saved_airline_df.count()
airport_count = saved_airport_df.count()

print("========== FINAL ROW COUNTS ==========")
print(f"Airline ID Table : {airline_count}")
print(f"Airport ID Table : {airport_count}")
print(f"Total Records    : {airline_count + airport_count}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from functools import reduce

CATALOG = "capstone_3flight"
SCHEMA = "flight_schema"
VOLUME = "ext_capstone3_vol"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
PRIMARY_PATH = f"{BASE_PATH}/primary_events/"
STATUS_PATH = f"{BASE_PATH}/status_updates/"
SECONDARY_PATH = f"{BASE_PATH}/secondary_events/"

AUTOLOADER_SCHEMA_PATH = f"{BASE_PATH}/autoloader_schema/primary_events"
PRIMARY_CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze_flight_events"

BRONZE_FLIGHT_TABLE = f"{CATALOG}.{SCHEMA}.bronze_flight_events"
BRONZE_CDC_TABLE = f"{CATALOG}.{SCHEMA}.bronze_status_cdc"
BRONZE_SECONDARY_TABLE = f"{CATALOG}.{SCHEMA}.bronze_secondary_events"

AIRLINE_TABLE = f"{CATALOG}.{SCHEMA}.airline"
AIRPORT_TABLE = f"{CATALOG}.{SCHEMA}.airport"

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("Catalog:", CATALOG)
print("Schema:", SCHEMA)
print("Volume root:", BASE_PATH)

In [0]:
required_source_folders = {
    "primary_events": PRIMARY_PATH,
    "status_updates": STATUS_PATH,
    "secondary_events": SECONDARY_PATH,
}

for folder_name, folder_path in required_source_folders.items():
    try:
        entries = dbutils.fs.ls(folder_path)
        csv_files = [x for x in entries if x.name.lower().endswith(".csv")]
        print(f"{folder_name}: {len(csv_files)} CSV file(s)")
        for item in csv_files:
            print("   ", item.name, item.size, "bytes")
    except Exception as e:
        print(f"ERROR reading {folder_name} at {folder_path}: {e}")
        raise

dbutils.fs.mkdirs(AUTOLOADER_SCHEMA_PATH)
dbutils.fs.mkdirs(PRIMARY_CHECKPOINT_PATH)


In [0]:
primary_file_infos = [
    x for x in dbutils.fs.ls(PRIMARY_PATH)
    if x.name.lower().endswith(".csv")
]
primary_file_infos = sorted(primary_file_infos, key=lambda x: x.name)

print("Primary CSV file count:", len(primary_file_infos))
display(spark.createDataFrame(
    [(x.name, x.path, x.size) for x in primary_file_infos],
    ["file_name", "file_path", "size_bytes"]
))

primary_dfs = []
primary_file_counts = []

for file_info in primary_file_infos:
    one_file_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .load(file_info.path)
        .withColumn("_source_file", F.lit(file_info.path))
    )
    primary_dfs.append(one_file_df)
    primary_file_counts.append(
        (file_info.name, one_file_df.count(), len(one_file_df.columns))
    )

if not primary_dfs:
    raise ValueError(f"No CSV files found in {PRIMARY_PATH}")

primary_raw = reduce(
    lambda left, right: left.unionByName(right, allowMissingColumns=True),
    primary_dfs
)

print("Primary total rows:", primary_raw.count())
print("Combined columns:", len(primary_raw.columns))
print("Combined column names:", primary_raw.columns)
display(primary_raw.limit(10))

In [0]:
file_count_df = spark.createDataFrame(
    primary_file_counts,
    ["file_name", "row_count", "column_count"]
).orderBy("file_name")

display(file_count_df)

print("Files with 13 columns (schema-evolved batches):")
display(file_count_df.filter(F.col("column_count") == 13))

print("Primary schema:")
primary_raw.printSchema()

In [0]:
schema_evolution_report = spark.createDataFrame(
    [
        (
            x.name,
            "source_schema_version" in next(
                d.columns for d, info in zip(primary_dfs, primary_file_infos)
                if info.name == x.name
            )
        )
        for x in primary_file_infos
    ],
    ["file_name", "has_source_schema_version"]
).orderBy("file_name")

display(schema_evolution_report)

if "source_schema_version" in primary_raw.columns:
    display(
        primary_raw
        .groupBy("_source_file", "source_schema_version")
        .count()
        .orderBy("_source_file", "source_schema_version")
    )
else:
    print("WARNING: source_schema_version was not detected in the combined source data.")

In [0]:
base_columns = [
    "flight_id", "event_timestamp", "ingest_timestamp",
    "airline_id", "airport_id", "flight_type", "flight_status",
    "timezone", "fare_amount", "delay_minutes", "aircraft_id",
    "source_system"
]

missing_base_columns = [c for c in base_columns if c not in primary_raw.columns]
if missing_base_columns:
    raise ValueError(f"Missing expected primary columns: {missing_base_columns}")

primary_total_rows = primary_raw.count()
primary_unique_flight_ids = primary_raw.select("flight_id").distinct().count()

duplicate_flight_id_groups = (
    primary_raw.groupBy("flight_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("flight_id")
)

exact_duplicate_groups = (
    primary_raw.groupBy(*base_columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_extra_rows = (
    exact_duplicate_groups
    .select(F.sum(F.col("count") - F.lit(1)).alias("duplicate_extra_rows"))
    .first()["duplicate_extra_rows"]
)

print("Total primary rows:", primary_total_rows)
print("Distinct flight IDs:", primary_unique_flight_ids)
print("Flight IDs occurring more than once:", duplicate_flight_id_groups.count())
print("Exact duplicate groups:", exact_duplicate_groups.count())
print("Extra duplicate rows:", duplicate_extra_rows)

display(duplicate_flight_id_groups)
display(exact_duplicate_groups)

In [0]:
null_profile = primary_raw.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in primary_raw.columns
])
display(null_profile)

print(
    "flight_type null count:",
    primary_raw.filter(F.col("flight_type").isNull()).count()
)

if "source_schema_version" in primary_raw.columns:
    print(
        "source_schema_version null count (expected on older batches):",
        primary_raw.filter(F.col("source_schema_version").isNull()).count()
    )

In [0]:
display(
    primary_raw
    .groupBy("flight_status")
    .count()
    .orderBy("flight_status")
)

In [0]:
primary_profile_typed = (
    primary_raw
    .withColumn("fare_amount_num", F.col("fare_amount").cast("double"))
    .withColumn("delay_minutes_num", F.col("delay_minutes").cast("double"))
)

display(
    primary_profile_typed
    .select(
        F.min("fare_amount_num").alias("min_fare"),
        F.max("fare_amount_num").alias("max_fare"),
        F.avg("fare_amount_num").alias("avg_fare"),
        F.min("delay_minutes_num").alias("min_delay"),
        F.max("delay_minutes_num").alias("max_delay"),
        F.avg("delay_minutes_num").alias("avg_delay")
    )
)

outlier_fares = (
    primary_profile_typed
    .filter(F.col("fare_amount_num") > 5000)
    .select("flight_id", "fare_amount", "delay_minutes", "_source_file")
    .orderBy(F.col("fare_amount_num").desc())
)

print("Fare values greater than 5000:", outlier_fares.count())
display(outlier_fares)

In [0]:
display(
    primary_raw
    .groupBy("timezone")
    .count()
    .orderBy(F.desc("count"))
)

display(
    primary_raw
    .select("event_timestamp", "ingest_timestamp")
    .orderBy("event_timestamp")
    .limit(10)
)

print("Distinct timezone values:", primary_raw.select("timezone").distinct().count())
print(
    "Rows where timezone = USD:",
    primary_raw.filter(F.upper(F.col("timezone")) == "USD").count()
)

In [0]:
status_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mode", "PERMISSIVE")
    .load(STATUS_PATH)
)

print("CDC row count:", status_raw.count())
print("CDC columns:", status_raw.columns)
status_raw.printSchema()
display(status_raw.limit(10))

In [0]:
display(
    status_raw
    .groupBy("new_status")
    .count()
    .orderBy("new_status")
)

cdc_total = status_raw.count()
cdc_distinct_sequences = status_raw.select("source_sequence").distinct().count()
cdc_duplicate_sequences = (
    status_raw.groupBy("source_sequence")
    .count()
    .filter(F.col("count") > 1)
)

print("CDC rows:", cdc_total)
print("Distinct source_sequence values:", cdc_distinct_sequences)
print("Duplicate source_sequence groups:", cdc_duplicate_sequences.count())
display(cdc_duplicate_sequences)


In [0]:
secondary_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("mode", "PERMISSIVE")
    .load(SECONDARY_PATH)
)

print("Secondary event rows:", secondary_raw.count())
print("Secondary columns:", secondary_raw.columns)
secondary_raw.printSchema()
display(secondary_raw.limit(10))

In [0]:
airline_df = spark.table(AIRLINE_TABLE)
airport_df = spark.table(AIRPORT_TABLE)

airline_count = airline_df.count()
airport_count = airport_df.count()

print("Airline table:", AIRLINE_TABLE, "| rows:", airline_count)
print("Airport table:", AIRPORT_TABLE, "| rows:", airport_count)

print("Airline schema:")
airline_df.printSchema()
print("Airport schema:")
airport_df.printSchema()

display(airline_df.limit(10))
display(airport_df.limit(10))

In [0]:
for table_name, df, key_col in [
    (AIRLINE_TABLE, airline_df, "airline_id"),
    (AIRPORT_TABLE, airport_df, "airport_id")
]:
    print(f"\nValidation for {table_name}")
    print("Rows:", df.count())
    print("Distinct keys:", df.select(key_col).distinct().count())
    print("Null keys:", df.filter(F.col(key_col).isNull()).count())
    print("Duplicate key groups:",
          df.groupBy(key_col).count().filter(F.col("count") > 1).count())

In [0]:
EXPECTED_PRIMARY_ROWS = 11540
EXPECTED_UNIQUE_FLIGHTS = 11500
EXPECTED_DUPLICATE_ROWS = 40
EXPECTED_FLIGHT_TYPE_NULLS = 12
EXPECTED_OUTLIER_FARES = 35
EXPECTED_CDC_ROWS = 520
EXPECTED_SECONDARY_ROWS = 310
EXPECTED_AIRLINE_ROWS = 110
EXPECTED_AIRPORT_ROWS = 150

In [0]:
def qa_status(actual, expected):
    return "PASS" if actual == expected else "CHECK"

observed_checks = [
    ("Primary rows", primary_total_rows, EXPECTED_PRIMARY_ROWS),
    ("Distinct flight IDs", primary_unique_flight_ids, EXPECTED_UNIQUE_FLIGHTS),
    ("Extra exact duplicate rows", int(duplicate_extra_rows or 0), EXPECTED_DUPLICATE_ROWS),
    ("Null flight_type rows", primary_raw.filter(F.col("flight_type").isNull()).count(), EXPECTED_FLIGHT_TYPE_NULLS),
    ("Fare outliers > 5000", outlier_fares.count(), EXPECTED_OUTLIER_FARES),
    ("CDC rows", cdc_total, EXPECTED_CDC_ROWS),
    ("Secondary event rows", secondary_raw.count(), EXPECTED_SECONDARY_ROWS),
    ("Airline reference rows", airline_count, EXPECTED_AIRLINE_ROWS),
    ("Airport reference rows", airport_count, EXPECTED_AIRPORT_ROWS),
]

reconciliation_df = spark.createDataFrame(
    [(name, int(actual), int(expected), qa_status(actual, expected))
     for name, actual, expected in observed_checks],
    ["check_name", "actual_value", "expected_value", "result"]
)

display(reconciliation_df)

In [0]:
primary_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaLocation", AUTOLOADER_SCHEMA_PATH)
    .option("rescuedDataColumn", "_rescued_data")
    .load(PRIMARY_PATH)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

primary_bronze_query = (
    primary_stream.writeStream
    .format("delta")
    .option("checkpointLocation", PRIMARY_CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .toTable(BRONZE_FLIGHT_TABLE)
)

primary_bronze_query.awaitTermination()

print("Auto Loader primary ingestion completed.")
print("Bronze table:", BRONZE_FLIGHT_TABLE)

In [0]:
bronze_flights = spark.table(BRONZE_FLIGHT_TABLE)

bronze_primary_count = bronze_flights.count()
bronze_distinct_flights = bronze_flights.select("flight_id").distinct().count()

print("Bronze flight rows:", bronze_primary_count)
print("Distinct flight IDs:", bronze_distinct_flights)
print("Columns:", bronze_flights.columns)
bronze_flights.printSchema()

display(
    bronze_flights
    .groupBy("_source_file")
    .count()
    .orderBy("_source_file")
)

display(
    bronze_flights
    .select("_source_file", "_ingested_at")
    .limit(10)
)

In [0]:
print(
    "Bronze rows retained beyond distinct flight IDs:",
    bronze_primary_count - bronze_distinct_flights
)

print(
    "source_schema_version present:",
    "source_schema_version" in bronze_flights.columns
)

if "source_schema_version" in bronze_flights.columns:
    display(
        bronze_flights
        .groupBy("source_schema_version")
        .count()
        .orderBy("source_schema_version")
    )

In [0]:

def load_static_csv_to_bronze(source_path, target_table):

    source_df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("mode", "PERMISSIVE")
        .load(source_path)
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    (
        source_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    loaded_df = spark.table(target_table)

    print(f"{target_table}: {loaded_df.count()} rows")

    return loaded_df


# Load CDC and secondary events into Bronze

bronze_cdc = load_static_csv_to_bronze(
    STATUS_PATH,
    BRONZE_CDC_TABLE
)

bronze_secondary = load_static_csv_to_bronze(
    SECONDARY_PATH,
    BRONZE_SECONDARY_TABLE
)


In [0]:
bronze_validation_rows = [
    ("bronze_flight_events", spark.table(BRONZE_FLIGHT_TABLE).count(), EXPECTED_PRIMARY_ROWS),
    ("bronze_status_cdc", spark.table(BRONZE_CDC_TABLE).count(), EXPECTED_CDC_ROWS),
    ("bronze_secondary_events", spark.table(BRONZE_SECONDARY_TABLE).count(), EXPECTED_SECONDARY_ROWS),
    ("airline", spark.table(AIRLINE_TABLE).count(), EXPECTED_AIRLINE_ROWS),
    ("airport", spark.table(AIRPORT_TABLE).count(), EXPECTED_AIRPORT_ROWS),
]

bronze_validation_df = spark.createDataFrame(
    [(name, int(actual), int(expected), qa_status(actual, expected))
     for name, actual, expected in bronze_validation_rows],
    ["table_name", "actual_rows", "expected_rows", "result"]
)

display(bronze_validation_df)

In [0]:
display(
    bronze_cdc.select(
        "_source_file",
        "_ingested_at"
    ).distinct()
)

In [0]:
display(
    bronze_secondary.select(
        "_source_file",
        "_ingested_at"
    ).distinct()
)

In [0]:
day1_summary = [
    ("Primary profiling", primary_total_rows, EXPECTED_PRIMARY_ROWS),
    ("Primary Bronze", bronze_primary_count, EXPECTED_PRIMARY_ROWS),
    ("CDC Bronze", spark.table(BRONZE_CDC_TABLE).count(), EXPECTED_CDC_ROWS),
    ("Secondary Bronze", spark.table(BRONZE_SECONDARY_TABLE).count(), EXPECTED_SECONDARY_ROWS),
    ("Airline reference", airline_count, EXPECTED_AIRLINE_ROWS),
    ("Airport reference", airport_count, EXPECTED_AIRPORT_ROWS),
]

display(spark.createDataFrame(
    [(name, int(actual), int(expected), qa_status(actual, expected))
     for name, actual, expected in day1_summary],
    ["component", "actual_rows", "expected_rows", "result"]
))

In [0]:
%sql
DESCRIBE TABLE capstone_3flight.flight_schema.bronze_flight_events;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(_source_file) AS source_file_count,
    COUNT(_ingested_at) AS ingestion_timestamp_count
FROM capstone_3flight.flight_schema.bronze_flight_events;

In [0]:
bronze_table = "capstone_3flight.flight_schema.bronze_flight_events"

bronze_df = spark.table(bronze_table)

print("Does source_schema_version exist?",
      "source_schema_version" in bronze_df.columns)

print("Total columns:", len(bronze_df.columns))

display(bronze_df.printSchema())

In [0]:
from pyspark.sql import functions as F

display(
    bronze_df
    .withColumn(
        "batch_file",
        F.regexp_extract(
            F.col("_source_file"),
            r"(flight_batch_\d+\.csv)",
            1
        )
    )
    .groupBy("batch_file", "source_schema_version")
    .count()
    .orderBy("batch_file")
)

In [0]:
BRONZE_TABLE = "capstone_3flight.flight_schema.bronze_flight_events"

before_count = spark.table(BRONZE_TABLE).count()

print("Before replay:", before_count)

In [0]:
after_count = spark.table(BRONZE_FLIGHT_TABLE).count()

print("Before:", before_count)
print("After:", after_count)

assert before_count == after_count == 11540, \
    "Replay safety validation failed!"

print("PASS: Auto Loader replay safety verified.")